# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access full metadata for the dataset
metadata = dataset.metadata

# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and fields, referencing their `@id`s as required.

In [ ]:
# List and overview of record sets, fields, and columns (all referenced by @id)
record_sets = list(dataset.record_sets())
print("Available Record Sets:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']} - Name: {rs.get('name','')} - Description: {rs.get('description','')}")

# For demonstration, show first record_set's fields info
if record_sets:
    first_rs_id = record_sets[0]['@id']
    fields = dataset.record_set_fields(record_set=first_rs_id)
    print(f"\nFields in RecordSet @id {first_rs_id}:")
    for field in fields:
        print(f"  - Field @id: {field['@id']} | name: {field.get('name','')} | dataType: {field.get('dataType','')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Extract data from each record set
dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df

        print(f"\nDataFrame for RecordSet @id {rs_id}")
        print("Columns:")
        print(df.columns.tolist())
        print("Sample records:")
        print(df.head())

# Demo: pick first available dataframe to continue
if len(dataframes) > 0:
    primary_rs_id = list(dataframes.keys())[0]
    primary_df = dataframes[primary_rs_id]
else:
    primary_rs_id = None
    primary_df = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data using field `@id`s.

In [ ]:
# Choose a numeric field for demonstration (by @id), fallback to first numeric column
if primary_df is not None and not primary_df.empty:
    # Try to find a numeric field by data type from schema
    numeric_field_id = None
    for field in dataset.record_set_fields(record_set=primary_rs_id):
        if field.get('dataType','').lower() in ['integer','float','number']:
            numeric_field_id = field['@id']
            break
    if not numeric_field_id:
        # fallback to first column
        numeric_field_id = primary_df.columns[0]
    
    print(f"Using numeric field @id: {numeric_field_id}")
    
    # Filtering: e.g., values greater than threshold
    threshold = 10
    if numeric_field_id in primary_df.columns:
        filtered_df = primary_df[primary_df[numeric_field_id].astype(float) > threshold]
        print(f"Filtered records (where {numeric_field_id} > {threshold}):")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) / filtered_df[numeric_field_id].astype(float).std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by: try to find a categorical field (e.g., anatomical location or similar)
        group_field_id = None
        for field in dataset.record_set_fields(record_set=primary_rs_id):
            if field.get('dataType','').lower() == 'text':
                group_field_id = field['@id']
                break
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
else:
    print("No available numeric field for EDA or no data loaded.")

## 5. Visualization
Visualize distributions and relationships between fields using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple visualization: histogram of numeric field
if primary_df is not None and numeric_field_id in primary_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(primary_df[numeric_field_id].astype(float), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Visualization: relationship between numeric and group field
if primary_df is not None and group_field_id and group_field_id in primary_df.columns and numeric_field_id in primary_df.columns:
    plt.figure(figsize=(7,5))
    sns.boxplot(y=primary_df[numeric_field_id].astype(float), x=primary_df[group_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated loading and preliminary analysis of the FAIR^2 colorectal cancer survivorship dataset using mlcroissant.

- Used Croissant schema URL for FAIR automated loading
- Referenced all entities (record set, fields, columns) by their `@id`
- Performed filtering, normalization, and grouping based on schema fields
- Visualized distributions and relationships between clinical/pathological features

Further work can include deeper statistical analysis, clinical prediction modeling, and exploration of other fields using their `@id` from the Croissant schema.